# Workshop Setup

One-click provisioning for the SQLBits 2026 workshop **"Adapting to Microsoft Fabric: A SQL Server Practitioner's Way Forward"**.

## The easiest way to set it up

1. Create a workspace in Microsoft Fabric.
2. Create a new notebook in that workspace.
3. Copy the code cell below into the new notebook.
4. Run all cells.

## Another way

1. Create a workspace in Microsoft Fabric.
2. Download this notebook from GitHub.
3. Import the notebook into your workspace.
4. Run all cells.

What happens when you run it:

1. Two Lakehouses (`bronze`, `silver`) and one Warehouse (`gold`) are created in the current workspace. Existing items with the same name are reused, so the notebook is safe to re-run.
2. The workshop notebooks are pulled from GitHub into your workspace, each pre-bound to the `silver` lakehouse as their default.
3. The `DataBuilder` notebook is executed so the `customers` and `orders` source tables (plus the parquet + CSV file copies) are ready before the workshop starts.


In [ ]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

LAKEHOUSE_NAMES = ["bronze", "silver"]
WAREHOUSE_NAMES = ["gold"]

GITHUB_ACCOUNT = "https://raw.githubusercontent.com/ChristianHenrikReich/TheEngineersGuideToMicrosoftFabric_handsons/refs/heads/main"

# No external data files to seed, DataBuilder generates everything at runtime.
DATA_FILES = []

# (github path, default lakehouse to bind the imported notebook to)
NOTEBOOKS_WITH_DEFAULT_LAKEHOUSES = [
    (f"{GITHUB_ACCOUNT}/notebooks/module_0_1_DataBuilder.ipynb",          "silver"),
    (f"{GITHUB_ACCOUNT}/notebooks/module_1_1_TimeTravel.ipynb",           "silver"),
    (f"{GITHUB_ACCOUNT}/notebooks/module_1_2_Optimize.ipynb",             "silver"),
    (f"{GITHUB_ACCOUNT}/notebooks/module_2_1_Language.ipynb",             "silver"),
    (f"{GITHUB_ACCOUNT}/notebooks/module_2_2_Catalyst.ipynb",             "silver"),
    (f"{GITHUB_ACCOUNT}/notebooks/module_2_3_Optimizing.ipynb",           "silver"),
    (f"{GITHUB_ACCOUNT}/notebooks/module_3_1_restore_point.ipynb",        "silver"),
    (f"{GITHUB_ACCOUNT}/notebooks/module_3_2_Interoperability.ipynb",     "silver"),
]

MAX_WORKERS = 5

# ---------------------------------------------------------------------------
from concurrent.futures import ThreadPoolExecutor
import os
import requests

# sempy_labs is the supported Python path for Fabric Warehouse provisioning.
# notebookutils has lakehouse.create but no warehouse.create.
%pip install semantic-link-labs --quiet


def _create_or_get_lakehouses(lakehouse_names):
    print("Get or create Lakehouses")
    lakehouses = {}
    for name in lakehouse_names:
        try:
            info = notebookutils.lakehouse.get(name=name)
            print(f"  Found Lakehouse: {name}")
        except Exception:
            info = notebookutils.lakehouse.create(name)
            print(f"  Created Lakehouse: {name}")
        lakehouses[name] = info
    return lakehouses


def _create_or_get_warehouses(warehouse_names):
    from sempy_labs.warehouse import create_warehouse, list_warehouses
    print("Get or create Warehouses")
    existing = set(list_warehouses()["Warehouse Name"].tolist())
    for name in warehouse_names:
        if name in existing:
            print(f"  Found Warehouse: {name}")
        else:
            create_warehouse(warehouse=name, description=f"Workshop warehouse: {name}")
            print(f"  Created Warehouse: {name}")


def _get_from_git(url):
    print(f"  Fetching {url}")
    r = requests.get(url)
    r.raise_for_status()
    return r


def _save_file_from_git(url, save_path):
    content = _get_from_git(url).content
    print(f"  Writing {save_path}")
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    with open(save_path, "wb") as f:
        f.write(content)


def _save_notebook_from_git(url, default_lakehouse=""):
    content = _get_from_git(url).text
    name = os.path.splitext(os.path.basename(url))[0]
    try:
        notebookutils.notebook.create(
            name=name, content=content, defaultLakehouse=default_lakehouse
        )
        print(f"  Imported notebook: {name} (default lakehouse: {default_lakehouse or 'none'})")
    except Exception:
        notebookutils.notebook.updateDefinition(
            name=name, content=content, defaultLakehouse=default_lakehouse
        )
        print(f"  Updated notebook:  {name} (default lakehouse: {default_lakehouse or 'none'})")


# ---------------------------------------------------------------------------
# 1. Create / get the medallion lakehouses and the gold warehouse
# ---------------------------------------------------------------------------
lakehouses = _create_or_get_lakehouses(LAKEHOUSE_NAMES)
_create_or_get_warehouses(WAREHOUSE_NAMES)

# ---------------------------------------------------------------------------
# 2. (Optional) mount a lakehouse and seed data files from GitHub
# ---------------------------------------------------------------------------
if DATA_FILES:
    primary = LAKEHOUSE_NAMES[0]
    mount_name = primary.lower()
    print(f"Mounting {primary} to /{mount_name}")
    notebookutils.fs.mount(
        lakehouses[primary].properties["abfsPath"],
        f"/{mount_name}",
        {"fileCacheTimeout": 0, "timeout": 120},
    )
    mount_path = notebookutils.fs.getMountPath(f"/{mount_name}")
    for url, dest in DATA_FILES:
        _save_file_from_git(url, f"{mount_path}{dest}")

# ---------------------------------------------------------------------------
# 3. Import workshop notebooks into the workspace (parallel)
# ---------------------------------------------------------------------------
print("Importing workshop notebooks")
if MAX_WORKERS == 1:
    for url, lh in NOTEBOOKS_WITH_DEFAULT_LAKEHOUSES:
        _save_notebook_from_git(url, lh)
else:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        list(ex.map(lambda t: _save_notebook_from_git(*t), NOTEBOOKS_WITH_DEFAULT_LAKEHOUSES))

print("Provisioning complete")


## Generate source data

The imported `DataBuilder` notebook is bound to the **silver** lakehouse. Running it
here populates `customers`, `orders`, `Files/orders_parquet/`, and
`Files/raw/sales_orders.csv`.

Expect 10-20 minutes on a starter Spark pool.


In [ ]:
%run module_0_1_DataBuilder


## Done

Next steps for attendees:

* Open any `module_*` notebook in the workspace, they are ready and bound to `silver`.
* Open the **silver** lakehouse: `customers`, `orders`, and the raw files are waiting.
* The **bronze** lakehouse is empty, to be filled during the workshop.
* The **gold** warehouse is empty, populated by the Module 3 interoperability exercise.
